In [3]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from datetime import datetime

# 중요도 판별
import eli5 
from eli5.sklearn import PermutationImportance 
from xgboost import XGBClassifier
from xgboost import plot_importance

# PyTorch 관련 import
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows: 맑은 고딕
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 기호 깨짐 방지

print(torch.cuda.is_available())

True


In [4]:
train_val = pd.read_csv("train.csv")
test_X = pd.read_csv("test.csv")

display(train_val.head())
display(test_X.head())

,Mass_Pilot,Width,Aspect,Inch,Plant,Proc_Param1,Proc_Param2,Proc_Param3,Proc_Param4,Proc_Param5,...,p247,p248,p249,p250,p251,p252,p253,p254,p255,Class
0,False,245,40,19,Plant_6,465.00,631.85,1460.8,1985,2059,...,1.475031,1.473817,1.472950,1.474981,1.477011,1.479157,1.483289,1.487421,1.491552,Good
1,False,275,40,21,Plant_2,516.01,701.87,1621.1,2205,2291,...,1.320114,1.326607,1.333636,1.340665,1.347694,1.352602,1.355859,1.359115,1.362371,Good
2,False,225,45,17,Plant_7,416.99,583.14,1310.0,1832,1918,...,1.344303,1.346259,1.348215,1.350183,1.352166,1.354149,1.356112,1.358066,1.360021,Good
3,True,275,30,20,Plant_7,490.01,623.89,1539.4,1960,2040,...,1.389538,1.389953,1.393673,1.397393,1.401113,1.405803,1.411120,1.416437,1.421754,Good
4,True,265,45,21,Plant_2,516.01,714.61,1621.1,2245,2336,...,1.435297,1.431648,1.428000,1.424352,1.420704,1.417607,1.415256,1.412904,1.410553,Good


,ID,Mass_Pilot,Width,Aspect,Inch,Plant,Proc_Param1,Proc_Param2,Proc_Param3,Proc_Param4,...,p246,p247,p248,p249,p250,p251,p252,p253,p254,p255
0,ID_0,True,235,60,18,Plant_2,442.01,687.55,1388.6,2160,...,1.169287,1.166026,1.166735,1.168156,1.170798,1.172955,1.173441,1.173989,1.175112,1.176234
1,ID_1,False,245,40,19,Plant_2,464.99,631.85,1460.8,1985,...,1.615543,1.628728,1.643133,1.657539,1.654435,1.647163,1.639891,1.637400,1.635183,1.632966
2,ID_2,False,235,40,19,Plant_2,464.99,619.43,1460.8,1946,...,1.453175,1.451461,1.448824,1.446188,1.435310,1.423030,1.410750,1.367765,1.321797,1.275830
3,ID_3,True,245,45,19,Plant_2,464.99,646.17,1460.8,2030,...,1.234581,1.238925,1.248509,1.260011,1.271885,1.286289,1.300693,1.306279,1.304755,1.303231
4,ID_4,True,255,55,19,Plant_2,464.99,708.24,1460.8,2225,...,1.346540,1.348108,1.349677,1.352246,1.355194,1.358359,1.361780,1.364867,1.366732,1.368597


In [5]:
train_X = train_val.drop(columns=['Class'])
train_Y = train_val['Class'].apply(lambda x: 1 if x == 'NG' else 0)

cat_list = train_X.select_dtypes(include=['object', 'bool', 'category']).columns.tolist()
num_list = sorted(list(set(train_X.columns) - set(cat_list)))

print(f"범주형 변수 개수: {len(cat_list)}")
print(f"수치형 변수 개수: {len(num_list)}")

test_ID = test_X['ID']
test_X = test_X.drop(columns=['ID'])

print(f"test_X shape: {test_X.shape}")
print(f"test_ID shape: {test_ID.shape}")

범주형 변수 개수: 3
수치형 변수 개수: 795
test_X shape: (466, 798)
test_ID shape: (466,)


In [6]:
OE = OneHotEncoder(min_frequency=0.01, handle_unknown='infrequent_if_exist', sparse_output=False)
OE.fit(train_X[cat_list])
onehot_cols = OE.get_feature_names_out(cat_list)

# train 데이터 원핫인코딩
after_onehot = pd.DataFrame(OE.transform(train_X[cat_list]), columns=onehot_cols, index=train_X.index)
train_X = pd.concat([after_onehot, train_X[num_list]], axis=1)
# test 데이터 원핫인코딩
after_onehot = pd.DataFrame(OE.transform(test_X[cat_list]), columns=onehot_cols, index=test_X.index)
test_X = pd.concat([after_onehot, test_X[num_list]], axis=1)
#data split
train_X, val_X, train_Y, val_Y = train_test_split(train_X, train_Y, test_size=0.2, random_state=42)

In [7]:
from imblearn.over_sampling import ADASYN

print(f"train_X shape: {train_X.shape}")
print(f"train_Y shape: {train_Y.shape}")

print(f"증강 전 데이터 분포")
print(train_Y.value_counts())

adasyn = ADASYN(random_state=42)
train_X_res, train_Y_res = adasyn.fit_resample(train_X, train_Y)

print(f"train_X shape: {train_X_res.shape}")
print(f"train_Y shape: {train_Y_res.shape}")

print(f"증강 후후 데이터 분포")
print(train_Y_res.value_counts())

train_X shape: (576, 809)
train_Y shape: (576,)
증강 전 데이터 분포
Class
0    489
1     87
Name: count, dtype: int64
train_X shape: (985, 809)
train_Y shape: (985,)
증강 후후 데이터 분포
Class
1    496
0    489
Name: count, dtype: int64


In [8]:
# 수치형 데이터 전처리 (전체 데이터 사용)
target_scale_col = num_list[:-256*3]
pass_scale_col = num_list[-256*3:]

train_X_basic_num = train_X_res[target_scale_col]  # -256*3 제거하여 전체 데이터 사용
train_X_cat = train_X_res[onehot_cols]
train_X_basic_xyp = train_X_res[pass_scale_col]

scaler = StandardScaler()
scaler.fit(train_X_basic_num)
train_X_basic_num_scaled = scaler.transform(train_X_basic_num)

train_X_basic_num_scaled_df = pd.DataFrame(
    train_X_basic_num_scaled, 
    columns=target_scale_col,
    index=train_X_res.index
)

train_X_combined = pd.concat([train_X_cat, train_X_basic_num_scaled_df, train_X_basic_xyp], axis=1)

print(f"원본 train_X shape: {train_X_res.shape}")
print(f"결합된 train_X_combined shape: {train_X_combined.shape}")

print(f"전체 컬럼 수: {len(train_X_combined.columns)}")
print(f"스케일링 적용 컬럼 수: {len(target_scale_col)}")
print(f"제외된 컬럼 수: {len(pass_scale_col)}")

print(train_X_combined.head())
print(train_X_combined.tail())



원본 train_X shape: (985, 809)
결합된 train_X_combined shape: (985, 809)
전체 컬럼 수: 809
스케일링 적용 컬럼 수: 27
제외된 컬럼 수: 768
   Mass_Pilot_False  Mass_Pilot_True  Plant_Plant_1  Plant_Plant_2  \
0               1.0              0.0            0.0            1.0   
1               1.0              0.0            0.0            0.0   
2               1.0              0.0            0.0            0.0   
3               1.0              0.0            0.0            0.0   
4               0.0              1.0            0.0            1.0   

   Plant_Plant_3  Plant_Plant_4  Plant_Plant_6  Plant_Plant_7  Plant_Plant_8  \
0            0.0            0.0            0.0            0.0            0.0   
1            0.0            0.0            1.0            0.0            0.0   
2            0.0            1.0            0.0            0.0            0.0   
3            0.0            0.0            0.0            1.0            0.0   
4            0.0            0.0            0.0            0.0      

In [9]:
# val, test 데이터 전처리 (standard scaling)
val_X_basic_num = val_X[target_scale_col]
val_X_basic_num_scaled = scaler.transform(val_X_basic_num)
val_X_basic_num_scaled_df = pd.DataFrame(
    val_X_basic_num_scaled, 
    columns=target_scale_col,
    index=val_X.index
)

val_X_combined = pd.concat([val_X[onehot_cols], val_X_basic_num_scaled_df, val_X[pass_scale_col]], axis=1)


test_X_basic_num = test_X[target_scale_col]
test_X_basic_num_scaled = scaler.transform(test_X_basic_num)
test_X_basic_num_scaled_df = pd.DataFrame(
    test_X_basic_num_scaled, 
    columns=target_scale_col,
    index=test_X.index
)

test_X_combined = pd.concat([test_X[onehot_cols], test_X_basic_num_scaled_df, test_X[pass_scale_col]], axis=1)

print(f"원본 val_X shape: {val_X.shape}")
print(f"원본 test_X shape: {test_X.shape}")
print(f"val_X_combined shape: {val_X_combined.shape}")
print(f"test_X_combined shape: {test_X_combined.shape}")

원본 val_X shape: (144, 809)
원본 test_X shape: (466, 809)
val_X_combined shape: (144, 809)
test_X_combined shape: (466, 809)


In [11]:
# analyze coordinate range
x_cols = [f'x{i}' for i in range(256)]
y_cols = [f'y{i}' for i in range(256)]

train_x_value = train_X_combined[x_cols].values.flatten()
train_y_value = train_X_combined[y_cols].values.flatten()
val_x_value = val_X[x_cols].values.flatten()
val_y_value = val_X[y_cols].values.flatten()
test_x_value = test_X[x_cols].values.flatten()
test_y_value = test_X[y_cols].values.flatten()

x_value = np.concatenate([train_x_value, val_x_value, test_x_value])
y_value = np.concatenate([train_y_value, val_y_value, test_y_value])

x_min = np.min(x_value)
x_max = np.max(x_value)
y_min = np.min(y_value)
y_max = np.max(y_value)

print(f"x 최소값: {x_min}, x 최대값: {x_max}")
print(f"y 최소값: {y_min}, y 최대값: {y_max}")



x 최소값: 160.57951, x 최대값: 425.6112
y 최소값: 40.519974, y 최대값: 185.615167612361


In [12]:
# ----------------------------------------------------
# 좌표 래스터화
# ----------------------------------------------------
class SpatialRasterizer:
    def __init__(self, x_min, x_max, y_min, y_max, grid_size=64):
        self.x_min = x_min
        self.x_max = x_max
        self.y_min = y_min
        self.y_max = y_max
        self.grid_size = grid_size
        self.x_range = x_max - x_min if x_max > x_min else 1
        self.y_range = y_max - y_min if y_max > y_min else 1

    def rasterize_with_real_coordinates(self, data_row):
        x_cols = [f'x{i}' for i in range(256)]
        y_cols = [f'y{i}' for i in range(256)]
        p_cols = [f'p{i}' for i in range(256)]

        x_coords = data_row[x_cols].values
        y_coords = data_row[y_cols].values
        p_values = data_row[p_cols].values

        grid = np.zeros((self.grid_size, self.grid_size), dtype=np.float32)
        count_grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int8)

        for i in range(256):
            if not (np.isnan(x_coords[i]) or np.isnan(y_coords[i]) or np.isnan(p_values[i])):
                x_norm = (x_coords[i] - self.x_min) / self.x_range
                y_norm = (y_coords[i] - self.y_min) / self.y_range
                x_idx = int(np.clip(x_norm * (self.grid_size - 1), 0, self.grid_size - 1))
                y_idx = int(np.clip(y_norm * (self.grid_size - 1), 0, self.grid_size - 1))
                grid[y_idx, x_idx] += p_values[i]
                count_grid[y_idx, x_idx] += 1

        mask = count_grid > 0
        grid[mask] = grid[mask] / count_grid[mask]

        return grid

# ----------------------------------------------------
# Feature Encoder (CNN + MLP)
# ----------------------------------------------------

class ImageCNN(nn.Module):
    def __init__(self, output_dim=64, input_size=64):
        super(ImageCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.conv4 = nn.Conv2d(128, 256, 3, padding=1)

        self.batch_norm1 = nn.BatchNorm2d(32)
        self.batch_norm2 = nn.BatchNorm2d(64)
        self.batch_norm3 = nn.BatchNorm2d(128)
        self.batch_norm4 = nn.BatchNorm2d(256)

        final_size = input_size // 16
        self.fc1 = nn.Linear(256 * final_size * final_size, 512)
        self.fc_out = nn.Linear(512, output_dim)

        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.batch_norm1(self.conv1(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm2(self.conv2(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm3(self.conv3(x))), 2)
        x = F.max_pool2d(F.relu(self.batch_norm4(self.conv4(x))), 2)

        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc_out(x)

class FeatureEncoder(nn.Module):
    """
    Feature Encoder:
      - ImageCNN: rasterized 좌표/압력 이미지 -> image embedding
      - basic MLP: tabular 기본 피처 -> basic embedding
      - concat 후 head까지 통과시켜 logit 출력
    """

    def __init__(self, basic_feature_dim, image_cnn_output_dim=64, basic_mlp_output_dim=32, input_grid_size=64):
        super(FeatureEncoder, self).__init__()

        self.image_cnn = ImageCNN(output_dim=image_cnn_output_dim, input_size=input_grid_size)

        self.basic_mlp = nn.Sequential(
            nn.Linear(basic_feature_dim, basic_feature_dim * 2),
            nn.ReLU(),
            nn.BatchNorm1d(basic_feature_dim * 2),
            nn.Dropout(0.3),
            nn.Linear(basic_feature_dim * 2, basic_mlp_output_dim),
            nn.ReLU()
        )

        combined_dim = image_cnn_output_dim + basic_mlp_output_dim
        self.head = nn.Sequential(
            nn.Linear(combined_dim, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )

    def forward(self, x_image, x_basic):
        img_feat = self.image_cnn(x_image)
        basic_feat = self.basic_mlp(x_basic)
        combined = torch.cat((img_feat, basic_feat), dim=1)
        output = self.head(combined)
        return output

    def extract_cnn_features(self, x_image):
        self.image_cnn.eval()
        
        with torch.no_grad():
            img_feat = self.image_cnn(x_image)
        return img_feat

# ----------------------------------------------------
# Dataset
# ----------------------------------------------------

class MultiModalDataset(Dataset):
    def __init__(self, full_df, basic_features_np, rasterizer, labels_np=None):
        self.full_df = full_df.reset_index(drop=True)
        self.basic_features_np = basic_features_np
        self.rasterizer = rasterizer
        self.labels_np = labels_np
        self.is_test = (labels_np is None)

    def __len__(self):
        return len(self.full_df)

    def __getitem__(self, idx):
        data_row = self.full_df.iloc[idx]
        image_grid = self.rasterizer.rasterize_with_real_coordinates(data_row)
        image_tensor = torch.from_numpy(image_grid).unsqueeze(0)  # (1, 64, 64)
        basic_feat_tensor = torch.from_numpy(self.basic_features_np[idx]).float()

        if self.is_test:
            return image_tensor, basic_feat_tensor
        else:
            label_tensor = torch.tensor(self.labels_np[idx], dtype=torch.float32).view(1)
            return image_tensor, basic_feat_tensor, label_tensor

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

train_X = train_X_combined
train_X = train_X.reset_index(drop=True)
train_X_basic = train_X.iloc[:, :-256*3].values
val_X = val_X_combined
val_X = val_X.reset_index(drop=True)
val_X_basic = val_X.iloc[:, :-256*3].values
test_X = test_X_combined
test_X = test_X.reset_index(drop=True)
test_X_basic = test_X.iloc[:, :-256*3].values

rasterizer = SpatialRasterizer(x_min, x_max, y_min, y_max)


train_data = MultiModalDataset(train_X, train_X_basic, rasterizer, train_Y_res.values)
val_data = MultiModalDataset(val_X, val_X_basic, rasterizer, val_Y.values)
test_data = MultiModalDataset(test_X, test_X_basic, rasterizer, None)


train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)
test_loader = DataLoader(test_data, batch_size=32, shuffle=False)


Using device: cuda


In [18]:
model = FeatureEncoder(basic_feature_dim=train_X_basic.shape[1], image_cnn_output_dim=64, basic_mlp_output_dim=32, input_grid_size=64)
model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-5)
criterion = nn.BCEWithLogitsLoss()

best_val_loss = float('inf')
best_epoch = 0

n_epochs = 30

os.makedirs("weights", exist_ok=True)
best_save_path = "weights/best_model.pth"

for epoch in range(n_epochs):
    model.train()
    train_loss_sum = 0.0
    for img, basic, y in train_loader:
        img = img.to(device)
        basic = basic.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(img, basic)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        
    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        for img, basic, y in val_loader:
            img = img.to(device)
            basic = basic.to(device)
            y = y.to(device)
            
            pred = model(img, basic)
            loss = criterion(pred, y)
            val_loss_sum += loss.item()
            
    avg_train_loss = train_loss_sum / len(train_loader)
    avg_val_loss = val_loss_sum / len(val_loader)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_epoch = epoch + 1
        torch.save(model.state_dict(), f'weights/best_model_{epoch}.pth')
        print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f} *best model*")
    else:
        print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

print(f"Best Epoch: {best_epoch}, Best Val Loss: {best_val_loss:.4f}")
torch.save(model.state_dict(), best_save_path)




Epoch 1 - Train Loss: 0.6622, Val Loss: 0.5898 *best model*
Epoch 2 - Train Loss: 0.5944, Val Loss: 0.5553 *best model*
Epoch 3 - Train Loss: 0.5573, Val Loss: 0.5607
Epoch 4 - Train Loss: 0.5067, Val Loss: 0.4971 *best model*
Epoch 5 - Train Loss: 0.4623, Val Loss: 0.5342
Epoch 6 - Train Loss: 0.3939, Val Loss: 0.5469
Epoch 7 - Train Loss: 0.3278, Val Loss: 0.6390
Epoch 8 - Train Loss: 0.2897, Val Loss: 0.6107
Epoch 9 - Train Loss: 0.2462, Val Loss: 0.5905
Epoch 10 - Train Loss: 0.1953, Val Loss: 0.6111
Epoch 11 - Train Loss: 0.1618, Val Loss: 0.8277
Epoch 12 - Train Loss: 0.1465, Val Loss: 0.5658
Epoch 13 - Train Loss: 0.1111, Val Loss: 0.7614
Epoch 14 - Train Loss: 0.0902, Val Loss: 0.6362
Epoch 15 - Train Loss: 0.0650, Val Loss: 0.6712
Epoch 16 - Train Loss: 0.0602, Val Loss: 0.7651
Epoch 17 - Train Loss: 0.0716, Val Loss: 0.6603
Epoch 18 - Train Loss: 0.0605, Val Loss: 0.6710
Epoch 19 - Train Loss: 0.0525, Val Loss: 0.8348
Epoch 20 - Train Loss: 0.0663, Val Loss: 0.6620
Epoch 21 -

In [19]:


#model.load_state_dict(torch.load(best_save_path))

model = FeatureEncoder(basic_feature_dim=train_X_basic.shape[1], image_cnn_output_dim=64, basic_mlp_output_dim=32, input_grid_size=64)
model.to(device)

#best_epoch = 10

final_model_save_path = "weights/final_model.pth"

full_data_x = pd.concat([train_X, val_X])
full_data_x_basic = full_data_x.iloc[:, :-256*3].values
full_data_y = pd.concat([train_Y_res, val_Y])

full_data = MultiModalDataset(full_data_x, full_data_x_basic, rasterizer, full_data_y.values)
full_loader = DataLoader(full_data, batch_size=32, shuffle=False)

for epoch in range(best_epoch):
    model.train()
    train_loss_sum = 0.0
    for img, basic, y in full_loader:
        img = img.to(device)
        basic = basic.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred = model(img, basic)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
    avg_train_loss = train_loss_sum / len(full_loader)
    print(f"Epoch {epoch+1} - Train Loss: {avg_train_loss:.4f}")

torch.save(model.state_dict(), final_model_save_path)
print(f"Final Model Saved to {final_model_save_path}")

print("Training complete")

Epoch 1 - Train Loss: 0.7730
Epoch 2 - Train Loss: 0.7591
Epoch 3 - Train Loss: 0.7792
Epoch 4 - Train Loss: 0.7700
Final Model Saved to weights/final_model.pth
Training complete


In [20]:
RF_model = RandomForestClassifier(n_estimators=200, random_state=1, n_jobs=-1)
model.eval()


features_list = []
labels_list = []

# Gradient 계산 비활성화 context
with torch.no_grad():
    for img, basic, y in full_loader:
        img = img.to(device)
        basic = basic.to(device)
        y = y.to(device)

        # 특징 추출
        features = model.extract_cnn_features(img)
        
        # [수정] .detach()를 붙여서 연산 그래프에서 분리 후 numpy 변환
        features_list.append(features.detach().cpu().numpy())
        labels_list.append(y.detach().cpu().numpy())

train_X_RF = pd.DataFrame(np.concatenate(features_list, axis=0))
train_Y_RF = pd.DataFrame(np.concatenate(labels_list, axis=0))

RF_model.fit(train_X_RF, train_Y_RF)

# test 데이터 특징 추출
features_list = []

with torch.no_grad():
    for img, basic in test_loader:
        img = img.to(device)
        basic = basic.to(device)
        y = y.to(device)

        features = model.extract_cnn_features(img)
        features_list.append(features.detach().cpu().numpy())
        labels_list.append(y.detach().cpu().numpy())

test_X_RF = pd.DataFrame(np.concatenate(features_list, axis=0))

pred = RF_model.predict_proba(test_X_RF)[:,1]



C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\base.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


In [21]:
submission = pd.read_csv("sample_submission.csv")
submission['probability'] = np.concatenate([pred,pred])

decision_id_L_list = submission.iloc[:466].sort_values('probability').iloc[:200]['ID']
decision_id_P_list = submission.iloc[466:].sort_values('probability').iloc[:200]['ID']

submission.loc[submission['ID'].isin(decision_id_L_list), 'decision'] = True
submission.loc[submission['ID'].isin(decision_id_P_list), 'decision'] = True

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
submission.to_csv(f"RF_aug_adasyn_submission_{timestamp}.csv", index=False)
display(submission)

,ID,probability,decision
0,ID_0_L,0.710,False
1,ID_1_L,0.255,True
2,ID_2_L,0.235,True
3,ID_3_L,0.370,False
4,ID_4_L,0.710,False
...,...,...,...
927,ID_461_P,0.165,True
928,ID_462_P,0.315,True
929,ID_463_P,0.475,False
930,ID_464_P,0.790,False
